# 09 — Bulk response validation: the IAF signature vs anti-TNF response

The single-cell result (notebooks 01-08): an inflammatory fibroblast (IAF) state expands in
disease, concentrated in a patient subgroup. Test here: does the state's transcriptional
program, scored in PRETREATMENT bulk biopsies, separate anti-TNF responders from
non-responders? Four public infliximab cohorts with mucosal-healing response labels:
GSE12251 (UC), GSE16879 (UC + CD), GSE23597 (UC), GSE73661 (UC, infliximab arm).

Two signatures: SIG9 = the 9-gene IAF program (IL11, IL13RA2, IL24, CHI3L1, MMP3, MMP10,
IL6, CXCL1, CXCL5); SIG25 = the 25-gene up-regulated set from the Martin IAF-vs-other
pseudobulk DE (notebook 08A). Score = mean per-gene z-score across the cohort subset.
Probe mapping: sig_probes.json (extracted from the GEO platform tables GPL570 / GPL6244).
Statistics: Mann-Whitney U, one-sided (NR > R); AUC with non-response as positive class.

VM-verified before this notebook was built (Nov 22, 2026); this run reproduces it.
Caveats: GSE23597 has only 7 non-responders; GSE16879 is analysed at biopsy level
(some patients contribute more than one biopsy); GSE73661 response is derived from the
same individual's week-4/6 Mayo endoscopic subscore (<=1 responder, >=2 non-responder).


In [ ]:
import gzip, io, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA = Path.home() / "Documents/immune-endotypes/data/bulk_response"
DATA.mkdir(parents=True, exist_ok=True)
MOD = Path.home() / "Documents/github/immune-endotypes-portfolio/modules/05_cross_cohort"
PROBES = json.loads((MOD / "sig_probes.json").read_text())

SIG9  = ["IL11","IL13RA2","IL24","CHI3L1","MMP3","MMP10","IL6","CXCL1","CXCL5"]
SIG25 = ["MMP1","SAA1","IL24","MASP1","WFDC2","C15orf48","CD82","TWIST2","SDC4","COL8A1",
         "SLC16A3","MME","CXCL5","CEMIP","CSF3","PTGS1","MMP3","SAA2","GK","TMEM132A",
         "CDCP1","ACKR4","SERINC2","AKR1B1","SQRDL"]
print("signatures:", len(SIG9), len(SIG25), "| platforms:", list(PROBES))


In [ ]:
URLS = {
 "GSE12251": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE12nnn/GSE12251/matrix/GSE12251_series_matrix.txt.gz",
 "GSE16879": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE16nnn/GSE16879/matrix/GSE16879_series_matrix.txt.gz",
 "GSE73661": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE73nnn/GSE73661/matrix/GSE73661_series_matrix.txt.gz",
 "GSE23597": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE23nnn/GSE23597/matrix/GSE23597_series_matrix.txt.gz",
}
for gse, url in URLS.items():
    out = DATA / f"{gse}_series_matrix.txt.gz"
    if not out.exists():
        subprocess.run(["curl", "-sSL", "-o", str(out), url], check=True)
    print(gse, out.stat().st_size // 10**6, "MB")


In [ ]:
def load_series(path):
    meta = {"title": [], "char": []}
    tbl = []
    with gzip.open(path, "rt", errors="replace") as f:
        in_tbl = False
        for line in f:
            if line.startswith("!series_matrix_table_begin"): in_tbl = True; continue
            if line.startswith("!series_matrix_table_end"): break
            if in_tbl: tbl.append(line); continue
            if line.startswith("!Sample_title"):
                meta["title"] = [v.strip('"') for v in line.rstrip("\n").split("\t")[1:]]
            elif line.startswith("!Sample_characteristics"):
                meta["char"].append([v.strip('"') for v in line.rstrip("\n").split("\t")[1:]])
    df = pd.read_csv(io.StringIO("".join(tbl)), sep="\t", index_col=0)
    df.columns = [c.strip('"') for c in df.columns]
    df.index = df.index.astype(str)
    return df, meta["title"], meta["char"]

def char_series(char_rows, key):
    for row in char_rows:
        if row[0].split(":")[0].strip().lower() == key:
            return pd.Series([r.split(":", 1)[1].strip() if ":" in r else r for r in row])
    return None

def gene_matrix(df, probe_map):
    inv = {}
    for g, probes in probe_map.items():
        ps = [p for p in probes if p in df.index]
        if ps: inv[g] = ps
    return pd.DataFrame({g: df.loc[ps].mean(axis=0) for g, ps in inv.items()}).T

def score_report(gm, labels, name, results):
    keep = labels.dropna(); gm = gm[keep.index]
    z = gm.sub(gm.mean(axis=1), axis=0).div(gm.std(axis=1) + 1e-9, axis=0)
    row = {"cohort": name, "n_R": int((keep == "R").sum()), "n_NR": int((keep == "NR").sum())}
    for sname, sig in [("SIG25", SIG25), ("SIG9", SIG9)]:
        genes = [g for g in sig if g in z.index]
        sc = z.loc[genes].mean(axis=0)
        r, nr = sc[keep == "R"], sc[keep == "NR"]
        U, _ = stats.mannwhitneyu(nr, r, alternative="two-sided")
        _, p = stats.mannwhitneyu(nr, r, alternative="greater")
        row[sname + "_genes"] = len(genes)
        row[sname + "_AUC"] = round(float(U / (len(nr) * len(r))), 3)
        row[sname + "_p"] = float(f"{p:.2e}")
        print(name, sname, "genes", len(genes), "AUC", row[sname + "_AUC"], "p", row[sname + "_p"])
    results.append(row)

results = []


In [ ]:
# GSE12251 (UC, GPL570): all samples are pretreatment (W0); label = week-8 response
df, titles, chars = load_series(DATA / "GSE12251_series_matrix.txt.gz")
resp = char_series(chars, "wk8rsphm")
labels = pd.Series({df.columns[i]: ("R" if resp[i] == "Yes" else "NR") for i in range(len(titles))})
print("GSE12251:", labels.value_counts().to_dict())
score_report(gene_matrix(df, PROBES["GPL570"]), labels, "GSE12251_UC", results)


In [ ]:
# GSE16879 (UC + CD, GPL570): biopsies before first infliximab; response label per patient
df, titles, chars = load_series(DATA / "GSE16879_series_matrix.txt.gz")
dis  = char_series(chars, "disease")
tim  = char_series(chars, "before or after first infliximab treatment")
resp = char_series(chars, "response to infliximab")
gm = gene_matrix(df, PROBES["GPL570"])
for strat in ["UC", "CD", "POOLED"]:
    lab = {}
    for i in range(len(titles)):
        if tim[i].startswith("Before") and resp[i] in ("Yes", "No") and (strat == "POOLED" or dis[i] == strat):
            lab[df.columns[i]] = "R" if resp[i] == "Yes" else "NR"
    print("GSE16879", strat, pd.Series(lab).value_counts().to_dict())
    score_report(gm, pd.Series(lab), f"GSE16879_{strat}", results)


In [ ]:
# GSE23597 (UC, GPL570): W0, non-placebo arms; label = wk8 response
df, titles, chars = load_series(DATA / "GSE23597_series_matrix.txt.gz")
tim  = char_series(chars, "time")
dose = char_series(chars, "dose")
resp = char_series(chars, "wk8 response")
lab = {}
for i in range(len(titles)):
    if tim[i] == "W0" and dose[i] != "placebo":
        lab[df.columns[i]] = "R" if resp[i] == "Yes" else "NR"
print("GSE23597:", pd.Series(lab).value_counts().to_dict())
score_report(gene_matrix(df, PROBES["GPL570"]), pd.Series(lab), "GSE23597_UC", results)


In [ ]:
# GSE73661 (UC, GPL6244): infliximab arm, W0 biopsy; response from the same individual's
# week-4/6 Mayo endoscopic subscore (<=1 responder, >=2 non-responder)
df, titles, chars = load_series(DATA / "GSE73661_series_matrix.txt.gz")
week  = char_series(chars, "week (w)")
ther  = char_series(chars, "induction therapy_maintenance therapy")
indiv = char_series(chars, "study individual number")
mayo  = char_series(chars, "mayo endoscopic subscore")
mayo_map = {indiv[i]: mayo[i] for i in range(len(titles)) if week[i] == "W4_W6" and ther[i] == "IFX"}
lab = {}
for i in range(len(titles)):
    if week[i] == "W0" and ther[i] == "IFX" and indiv[i] in mayo_map:
        lab[df.columns[i]] = "R" if mayo_map[indiv[i]] in ("0", "1") else "NR"
print("GSE73661:", pd.Series(lab).value_counts().to_dict())
score_report(gene_matrix(df, PROBES["GPL6244"]), pd.Series(lab), "GSE73661_UC_IFX", results)


In [ ]:
res = pd.DataFrame(results)
res.to_csv(MOD / "results/bulk_response_validation.csv", index=False)
print(res.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for ax, sig in zip(axes, ["SIG25_AUC", "SIG9_AUC"]):
    d = res.iloc[::-1]
    ax.barh(d["cohort"], d[sig], color="#4472C4")
    ax.axvline(0.5, color="grey", ls="--", lw=1)
    ax.set_xlim(0, 1); ax.set_title(sig.replace("_AUC", "") + " — AUC for non-response")
    for y, v in enumerate(d[sig]): ax.text(v + 0.02, y, f"{v:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(MOD / "figures/bulk_response_auc.png", dpi=200, bbox_inches="tight")
print("saved results/bulk_response_validation.csv + figures/bulk_response_auc.png")


In [ ]:
# Comparators + severity control (added Nov 23, 2026 after review):
# OSM alone (West 2017 benchmark) and a generic inflammation score INFLAM6
# (S100A8, S100A9, IL1B, CXCL8, LCN2, REG1A - disjoint from both signatures),
# plus the GSE73661 severity-matched stratum (baseline Mayo subscore = 3 only).
INFLAM = ["S100A8","S100A9","IL1B","CXCL8","LCN2","REG1A"]
def auc_of(gm, labels, genes):
    keep = labels.dropna(); g = gm[keep.index]
    z = g.sub(g.mean(axis=1), axis=0).div(g.std(axis=1) + 1e-9, axis=0)
    gs = [x for x in genes if x in z.index]
    sc = z.loc[gs].mean(axis=0) if len(gs) > 1 else z.loc[gs[0]]
    r, nr = sc[keep == "R"], sc[keep == "NR"]
    U, _ = stats.mannwhitneyu(nr, r, alternative="two-sided")
    return round(float(U / (len(nr) * len(r))), 3)
def cmp_block(name, gm, labels, rows):
    rows.append({"stratum": name, "n_R": int((labels=="R").sum()), "n_NR": int((labels=="NR").sum()),
                 "SIG25": auc_of(gm, labels, SIG25), "SIG9": auc_of(gm, labels, SIG9),
                 "OSM": auc_of(gm, labels, ["OSM"]), "INFLAM6": auc_of(gm, labels, INFLAM)})
    print(rows[-1])

cmp_rows = []
df, titles, chars = load_series(DATA / "GSE12251_series_matrix.txt.gz")
resp = char_series(chars, "wk8rsphm")
cmp_block("GSE12251_UC", gene_matrix(df, PROBES["GPL570"]),
          pd.Series({df.columns[i]: ("R" if resp[i]=="Yes" else "NR") for i in range(len(titles))}), cmp_rows)
df, titles, chars = load_series(DATA / "GSE16879_series_matrix.txt.gz")
dis = char_series(chars, "disease"); tim = char_series(chars, "before or after first infliximab treatment"); resp = char_series(chars, "response to infliximab")
gm = gene_matrix(df, PROBES["GPL570"])
for strat in ["UC","CD","POOLED"]:
    lab = {df.columns[i]: ("R" if resp[i]=="Yes" else "NR") for i in range(len(titles))
           if tim[i].startswith("Before") and resp[i] in ("Yes","No") and (strat=="POOLED" or dis[i]==strat)}
    cmp_block(f"GSE16879_{strat}", gm, pd.Series(lab), cmp_rows)
df, titles, chars = load_series(DATA / "GSE23597_series_matrix.txt.gz")
tim = char_series(chars, "time"); dose = char_series(chars, "dose"); resp = char_series(chars, "wk8 response")
cmp_block("GSE23597_UC", gene_matrix(df, PROBES["GPL570"]),
          pd.Series({df.columns[i]: ("R" if resp[i]=="Yes" else "NR") for i in range(len(titles)) if tim[i]=="W0" and dose[i]!="placebo"}), cmp_rows)
df, titles, chars = load_series(DATA / "GSE73661_series_matrix.txt.gz")
week = char_series(chars, "week (w)"); ther = char_series(chars, "induction therapy_maintenance therapy")
indiv = char_series(chars, "study individual number"); mayo = char_series(chars, "mayo endoscopic subscore")
mmap = {indiv[i]: mayo[i] for i in range(len(titles)) if week[i]=="W4_W6" and ther[i]=="IFX"}
lab = {}; bm = {}
for i in range(len(titles)):
    if week[i]=="W0" and ther[i]=="IFX" and indiv[i] in mmap:
        lab[df.columns[i]] = "R" if mmap[indiv[i]] in ("0","1") else "NR"; bm[df.columns[i]] = mayo[i]
labels = pd.Series(lab); gm = gene_matrix(df, PROBES["GPL6244"])
cmp_block("GSE73661_UC_IFX", gm, labels, cmp_rows)
cmp_block("GSE73661_Mayo3only", gm, labels[pd.Series(bm)=="3"], cmp_rows)
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(MOD / "results/bulk_response_comparators.csv", index=False)
print(cmp_df.to_string(index=False))


## Read-out (corrected Nov 23, 2026 — positioning: independent replication and extension)

Both signatures score higher in non-responders in EVERY cohort stratum (SIG25 AUC
0.71-0.93; SIG9 AUC 0.77-0.95). BUT the comparator cell shows this discrimination is
largely shared with generic inflammation: OSM alone reaches 0.985 in GSE12251 and
0.851 in GSE23597, and INFLAM6 is comparable overall (pooled 0.869 vs SIG9 0.879)
and higher in CD. Friedrich et al. (Nat Med 2021) already validated fibroblast /
neutrophil pathotype signatures for non-response across cohorts that include
GSE12251 and GSE16879 — so the bulk result is an independent replication of an
established association, NOT a new discovery, and the manuscript says so.

What survives the controls: within GSE73661 patients with identical baseline
endoscopic severity (Mayo 3, n=19), SIG9 keeps AUC 0.859 while SIG25 / OSM /
INFLAM6 fall to 0.68-0.71 — suggestive (one cohort, small n) that the compact
cell-state program is not only a severity readout. The paper's primary claims rest
on the single-cell analyses: uniform cross-cohort + cross-species quantification,
the patient-subgroup burden distribution, independence from GIMATS, and the
driver analysis (whose NF-kB signal is downstream of IL-1R as well as TNF —
compatible with Friedrich's IL-1R dependence; see Discussion when written).
